##
            TAVILY [WEB-SEARCH] 
                ||      Tool Call
                ||
                VV
    START -> CHATBOT -> END
                ^^
                ||
            LLM-Prompt

Making the websearch tool integration with TAVILY for web-search and state awarness

In [5]:
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from typing import Annotated
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


In [6]:
from langchain_ollama import ChatOllama


# Load local model
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0,
)

In [7]:
# Define the state
class ChatState(TypedDict):
    user_input: str
    response: str

In [9]:
# Node function
def chatbot(state: ChatState):
    answer = llm.invoke(state["user_input"])

    return {
        "response": answer.content
    }


In [10]:
# Build graph
builder = StateGraph(ChatState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()


In [11]:
# Run chatbot
while True:
    question = input("You: ")

    if question.lower() in ["exit", "quit"]:
        break

    result = graph.invoke({
        "user_input": question
    })

    print("Bot:", result["response"])

Bot: Hello! I’m doing great, thank you for asking. How can I help you today?


## Chat bot with tools

In [13]:
from langchain_tavily import TavilySearch

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

tavily_key = os.getenv("TAVILY_API_KEY")

print(tavily_key[:10])

tvly-dev-1


In [17]:
from langchain_tavily import TavilySearch

search = TavilySearch(
    max_results=5,
    topic="news"
)

In [18]:
results = search.invoke("Latest AI news")
print(results)

{'query': 'Latest AI news', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-may-2026', 'title': 'The latest AI news we announced in May 2026', 'content': "May 2026 was packed with AI news. At Google I/O 2026, we officially entered the agentic Gemini era with the launch of Gemini 3.5 — which delivers frontier intelligence for agents and coding — and Gemini Omni, where Gemini’s ability to reason meets the ability to create. The Android Show set the stage with brand-new hardware built specifically for these tools, including the Googlebook from our hardware partners. We also broadened our personal wellness tools with the new Google Health app and [...] Breadcrumb\n\n1. Home\n2. Innovation & AI\n3. Technology\n4. AI\n\n# The latest AI news we announced in May 2026\n\nJun 05, 2026\n\n|\n\n x.com\n Facebook\n LinkedIn\n Mail\n\nHere’s a recap of our biggest AI updates from May, including anno